# Guide 7 — Talking to the Referee

> **PYNQ Bootcamp guide.** This notebook takes one piece of the big competition program and explains it in small steps. Almost all of the code here is the *real* code that runs during a match — we've just split it up and added plain-English notes so it's easy to follow. (The one exception is the *Matching Strategy* guide, where the game plan is written as pseudocode for you to think through.)

## What is this notebook about?

The referee is another computer that runs the game. This guide is the "translator"
that sends and receives messages in exactly the format the referee expects — like
learning the exact words to say so the referee understands you.

This part has **no game strategy** in it at all. It just sends and receives. That's
on purpose: keeping it simple makes it easy to test. All real code.


### How this guide fits in

**Depends on:** Guide 1 (loads `pynqp2p`). **Used by:** Guide 8's brain, Guide 10, and the main turn loop.

*New here? Read **Guide 0 — How Everything Connects** first for the big picture.*


### The `RefereeClient`

Each function here sends one kind of message: flip two cards, report the result, ask
for a hint, or join the game. Notice how each message is just a small labeled note
(a dictionary) turned into text and sent. `poll` collects messages coming back.


In [ ]:
class RefereeClient:
    def __init__(self, server, key, referee_id, team, master_id=None, board_id=None):
        pynqp2p.register(server, key)
        self.referee_id = referee_id
        self.master_id = master_id
        self.team = team
        self.board_id = board_id or pynqp2p.get_id()

    def send(self, message):
        pynqp2p.send(self.referee_id, json.dumps(message))

    def poll(self):
        """Drain and parse every queued message. A malformed line is logged
        and skipped rather than raised, since one bad line should never take
        down the match loop."""
        messages = []
        for raw in pynqp2p.receive_all():
            try:
                messages.append(json.loads(raw))
            except json.JSONDecodeError:
                print(f'[referee] skipping malformed message: {raw!r}')
        return messages

    def flip_both(self, pos1, pos2):
        self.send({'type': 'flip_both', 'team': self.team, 'pos1': pos1, 'pos2': pos2})

    def report_result(self, pos1, pos2, cls1, cls2, claim):
        self.send({
            'type': 'report_result', 'team': self.team,
            'pos1': pos1, 'pos2': pos2, 'cls1': cls1, 'cls2': cls2, 'claim': claim,
        })

    def request_hint(self, obj):
        self.send({'type': 'hint_request', 'team': self.team, 'object': obj})

    def join_competition(self, secret):
        """Self-reports this board's MAC to the Master's lobby, once, right
        after connecting -- lets the operator's match-assign popup auto-fill
        your MAC instead of typing it in by hand. Requires `master_id` (does
        not change between rounds, unlike `referee_id`). Entirely optional:
        skip it and the operator can still enter your MAC manually."""
        if not self.master_id:
            print('[referee] join_competition skipped: no MASTER_ID set.')
            return
        lobby_id = f'{self.master_id}-lobby'
        pynqp2p.send(lobby_id, json.dumps({
            'type': 'join', 'team': self.team, 'mac': self.board_id, 'secret': secret,
        }))

### Check yourself

1. Why is it a good idea to keep all the game *strategy* out of this file?
2. What does `flip_both` do in one round-trip that two separate `flip` calls would
   do in two?
